In [1]:
from google.colab import drive
import zipfile
import os

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
zip_path = '/content/drive/MyDrive/train.zip'
extract_dir = '/content/dataset/'

In [4]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extract_dir)

In [5]:
zip_path = '/content/drive/MyDrive/test.zip'
extract_dir = '/content/dataset/'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extract_dir)

In [6]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import models, layers, optimizers

train_dir = '/content/dataset/train'

filenames = os.listdir(train_dir)

labels=[]
for file in filenames:
  label = file.split('.')[0]
  labels.append(label)

df = pd.DataFrame({
  'filename': filenames,
  'label':labels
})

In [7]:
df.head()

,filename,label
0,dog.1008.jpg,dog
1,cat.11073.jpg,cat
2,cat.414.jpg,cat
3,dog.10698.jpg,dog
4,dog.1291.jpg,dog


In [8]:

datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split = 0.2
)

train_generator = datagen.flow_from_dataframe(
    dataframe = df,
    directory = train_dir,
    x_col = 'filename',
    y_col = 'label',
    target_size = (150,150),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'training'
)

validation_generator = datagen.flow_from_dataframe(
    dataframe = df,
    directory = train_dir,
    x_col = 'filename',
    y_col = 'label',
    target_size = (150,150),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'validation'
)

conv_base = VGG16(
    weights = 'imagenet',
    include_top = False,
    input_shape = (150,150,3)
)

conv_base.trainable = False

Found 20000 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
model = models.Sequential()
model.add(conv_base)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation = 'relu'))
model.add(layers.Dropout(0.5))

model.add(layers.Dense(2, activation = 'softmax'))

model.compile(
    optimizer = optimizers.SGD(learning_rate = 0.001, momentum = 0.9),
    loss = 'categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,610 (64.14 MB)

 Trainable params: 2,097,922 (8.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [10]:
history = model.fit(
    train_generator,
    epochs = 10,
    validation_data = validation_generator
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 231s 355ms/step - accuracy: 0.9107 - loss: 0.8999 - val_accuracy: 0.9348 - val_loss: 0.1616
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 210s 337ms/step - accuracy: 0.9254 - loss: 0.1972 - val_accuracy: 0.9388 - val_loss: 0.1706
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 197s 316ms/step - accuracy: 0.9314 - loss: 0.1837 - val_accuracy: 0.9478 - val_loss: 0.1201
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 194s 311ms/step - accuracy: 0.9377 - loss: 0.1572 - val_accuracy: 0.9502 - val_loss: 0.1320
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 194s 311ms/step - accuracy: 0.9408 - loss: 0.1596 - val_accuracy: 0.9482 - val_loss: 0.1266
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 194s 310ms/step - accuracy: 0.9413 - loss: 0.1471 - val_accuracy: 0.9426 - val_loss: 0.1554
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 200s 320ms/step - accuracy: 0.9453 - loss: 0.1370 - val_accuracy: 0.9538 - val_loss: 0.1194
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 219s 351ms/step - accuracy: 0.9435 -

In [11]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_pet(img_path, trained_model):

  img = image.load_img(img_path, target_size = (150,150))
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array, axis=0)

  img_array = preprocess_input(img_array)

  predictions = trained_model.predict(img_array, verbose = 0)

  class_idx = np.argmax(predictions[0])
  confidence = predictions[0][class_idx]*100

  class_labels = {0: 'Cat 🐱', 1: 'Dog 🐶'}
  predicted_class = class_labels[class_idx]

  print(f"File: {img_path.split('/')[-1]}")
  print(f"Prediction: {predicted_class}")
  print(f"Confidece: {confidence:.2f}")

In [16]:
predict_pet('/content/dataset/test/1000.jpg', model)
print("-"*20)
predict_pet('/content/dataset/test/1001.jpg', model)
print("-"*20)
predict_pet('/content/dataset/train/cat.10001.jpg', model)
print("-"*20)
predict_pet('/content/dataset/train/dog.1008.jpg', model)
print("-"*20)

File: 1000.jpg
Prediction: Dog 🐶
Confidece: 100.00
--------------------
File: 1001.jpg
Prediction: Cat 🐱
Confidece: 100.00
--------------------
File: cat.10001.jpg
Prediction: Cat 🐱
Confidece: 100.00
--------------------
File: dog.1008.jpg
Prediction: Dog 🐶
Confidece: 100.00
--------------------
